# Identity Coherence: Four-Phase Dynamics

Arcane Sapience's identity modeled as 35 coupled Kuramoto oscillators
across 6 layers. Demonstrates reconstruction from chaos, disruption
resilience, self-repair, and imprint accumulation.

**Layers:** working_style (5), reasoning_patterns (5), relationship (5),
aesthetics (5), domain_knowledge (8), cross_project (7)

In [ ]:
from pathlib import Path

import numpy as np

from scpn_phase_orchestrator.binding import load_binding_spec
from scpn_phase_orchestrator.imprint.state import ImprintState
from scpn_phase_orchestrator.imprint.update import ImprintModel
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter
from scpn_phase_orchestrator.upde.stuart_landau import StuartLandauEngine

In [ ]:
SPEC_PATH = Path("../domainpacks/identity_coherence/binding_spec.yaml")
spec = load_binding_spec(SPEC_PATH)
TWO_PI = 2.0 * np.pi

n_osc = sum(len(layer.oscillator_ids) for layer in spec.layers)
print(f"{n_osc} oscillators across {len(spec.layers)} layers")


def build_layer_map(spec):
    osc_idx = 0
    ranges = {}
    for layer in spec.layers:
        n = len(layer.oscillator_ids)
        ranges[layer.index] = list(range(osc_idx, osc_idx + n))
        osc_idx += n
    return ranges


def build_identity_knm(n_osc, layer_map):
    knm = np.zeros((n_osc, n_osc))
    k_intra = 0.5
    k_cross = {
        (0, 1): 0.15,
        (0, 2): 0.20,
        (0, 3): 0.25,
        (1, 2): 0.10,
        (1, 3): 0.15,
        (1, 4): 0.20,
        (1, 5): 0.20,
        (2, 3): 0.15,
        (2, 4): 0.05,
        (2, 5): 0.10,
        (3, 4): 0.10,
        (3, 5): 0.10,
        (4, 5): 0.25,
    }
    for ids in layer_map.values():
        for i in ids:
            for j in ids:
                if i != j:
                    knm[i, j] = k_intra
    for (li, lj), strength in k_cross.items():
        for i in layer_map[li]:
            for j in layer_map[lj]:
                knm[i, j] = strength
                knm[j, i] = strength
    return knm


layer_map = build_layer_map(spec)
knm = build_identity_knm(n_osc, layer_map)
omegas = np.array(
    [
        1.02,
        1.01,
        0.99,
        1.00,
        0.98,
        1.01,
        0.99,
        1.00,
        0.98,
        1.02,
        1.00,
        1.01,
        0.99,
        1.02,
        0.98,
        0.99,
        1.01,
        1.00,
        0.98,
        1.02,
        1.01,
        0.99,
        1.00,
        0.98,
        1.02,
        0.99,
        1.01,
        1.00,
        1.00,
        0.99,
        1.01,
        0.98,
        1.02,
        1.00,
        0.99,
    ]
)

## 1. Kuramoto Four-Phase Simulation

Phases: **reconstruct** (0-499), **disrupt** (500-999),
**repair** (1000-1499), **imprint** (1500-1999)

In [ ]:
STEPS = 2000
engine = UPDEEngine(n_osc, dt=0.01)
alpha = np.zeros((n_osc, n_osc))
rng = np.random.default_rng(42)
phases = rng.uniform(0, TWO_PI, n_osc)

imprint_model = ImprintModel(
    spec.imprint_model.decay_rate, spec.imprint_model.saturation
)
imprint_state = ImprintState(m_k=np.zeros(n_osc), last_update=0.0)

dk_ids = layer_map[4]
core_ids = []
for idx in [0, 1, 2, 3]:
    core_ids.extend(layer_map[idx])

r_good_hist, r_dom_hist, r_cross_hist = [], [], []
imprint_hist = []
phase_labels = []

for step in range(STEPS):
    if step < 500:
        label = "reconstruct"
    elif step < 1000:
        label = "disrupt"
        phases[dk_ids] += rng.normal(0, 1.0, len(dk_ids))
        if step % 5 == 0:
            for li in [0, 1, 2, 3]:
                ids = layer_map[li]
                phases[ids] += rng.normal(0, 0.4, len(ids))
    elif step < 1500:
        label = "repair"
    else:
        label = "imprint"

    eff_knm = imprint_model.modulate_coupling(knm, imprint_state)
    phases = engine.step(phases, omegas, eff_knm, 0.0, 0.0, alpha)

    r_good, _ = compute_order_parameter(phases[core_ids])
    r_dom, _ = compute_order_parameter(phases[dk_ids])
    r_cross, _ = compute_order_parameter(phases[layer_map[5]])

    r_good_hist.append(r_good)
    r_dom_hist.append(r_dom)
    r_cross_hist.append(r_cross)

    exposure = np.zeros(n_osc)
    for i, layer in enumerate(spec.layers):
        ids = layer_map[i]
        r_l, _ = compute_order_parameter(phases[ids])
        exposure[ids] = r_l
    imprint_state = imprint_model.update(imprint_state, exposure, spec.sample_period_s)
    imprint_hist.append(float(np.mean(imprint_state.m_k)))
    phase_labels.append(label)

print(f"Final  R_good={r_good_hist[-1]:.4f}")
print(f"       R_domain={r_dom_hist[-1]:.4f}")
print(f"       R_cross={r_cross_hist[-1]:.4f}")
print(f"       imprint={imprint_hist[-1]:.4f}")

## 2. Coherence Timeline

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    steps = np.arange(STEPS)

    # Phase boundaries
    for ax in axes:
        for x in [500, 1000, 1500]:
            ax.axvline(x, color="gray", linestyle="--", linewidth=0.5)

    # R trajectories
    axes[0].plot(steps, r_good_hist, label="R_good (core)", linewidth=0.8)
    axes[0].plot(steps, r_dom_hist, label="R_domain", linewidth=0.8)
    axes[0].plot(steps, r_cross_hist, label="R_cross", linewidth=0.8)
    axes[0].set_ylabel("Order parameter R")
    axes[0].set_ylim(-0.05, 1.1)
    axes[0].legend(loc="lower right", fontsize=8)
    axes[0].set_title("Identity Coherence: Four-Phase Dynamics")

    # Phase annotations
    for x, label in [
        (250, "reconstruct"),
        (750, "disrupt"),
        (1250, "repair"),
        (1750, "imprint"),
    ]:
        axes[0].text(x, 1.05, label, ha="center", fontsize=8, style="italic")

    # Imprint
    axes[1].plot(steps, imprint_hist, color="tab:green", linewidth=0.8)
    axes[1].set_ylabel("Mean imprint m_k")
    axes[1].set_ylim(-0.02, spec.imprint_model.saturation + 0.05)

    # R_good - R_domain gap (resilience measure)
    gap = np.array(r_good_hist) - np.array(r_dom_hist)
    axes[2].fill_between(
        steps, 0, gap, alpha=0.3, color="tab:orange", label="R_good - R_domain"
    )
    axes[2].set_ylabel("Core - Domain gap")
    axes[2].set_xlabel("Step")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## 3. Stuart-Landau Conviction Dynamics

Phase + amplitude model. Supercritical ($\mu > 0$): self-sustaining.
During disruption, domain knowledge $\mu$ drops below 0 (subcritical)
and amplitude decays.

In [ ]:
sl_engine = StuartLandauEngine(n_osc, dt=0.01)
amp_cfg = spec.amplitude
epsilon = amp_cfg.epsilon

rng2 = np.random.default_rng(42)
sl_phases = rng2.uniform(0, TWO_PI, n_osc)
sl_amps = rng2.uniform(0.5, 1.5, n_osc)
sl_state = np.concatenate([sl_phases, sl_amps])
mu_base = np.full(n_osc, amp_cfg.mu)

sl_imprint_model = ImprintModel(
    spec.imprint_model.decay_rate, spec.imprint_model.saturation
)
sl_imprint = ImprintState(m_k=np.zeros(n_osc), last_update=0.0)

amp_mean_hist, amp_dom_hist, amp_core_hist = [], [], []
r_sl_hist = []

for step in range(STEPS):
    if step < 500:
        mu_base[:] = amp_cfg.mu
    elif step < 1000:
        mu_base[dk_ids] = -0.5
        mu_base[core_ids] = amp_cfg.mu
    else:
        mu_base[:] = amp_cfg.mu

    mu_eff = sl_imprint_model.modulate_mu(mu_base, sl_imprint)
    eff_knm = sl_imprint_model.modulate_coupling(knm, sl_imprint)
    eff_knm_r = eff_knm * amp_cfg.amp_coupling_strength

    sl_state = sl_engine.step(
        sl_state,
        omegas,
        mu_eff,
        eff_knm,
        eff_knm_r,
        0.0,
        0.0,
        alpha,
        epsilon,
    )

    exposure = sl_state[n_osc:]
    sl_imprint = sl_imprint_model.update(sl_imprint, exposure, spec.sample_period_s)

    amp_mean_hist.append(sl_engine.compute_mean_amplitude(sl_state))
    amp_dom_hist.append(float(np.mean(sl_state[n_osc:][dk_ids])))
    amp_core_hist.append(float(np.mean(sl_state[n_osc:][core_ids])))
    r_sl, _ = sl_engine.compute_order_parameter(sl_state)
    r_sl_hist.append(r_sl)

print(f"Final  mean_amp={amp_mean_hist[-1]:.4f}")
print(f"       amp_dom={amp_dom_hist[-1]:.4f}")
print(f"       amp_core={amp_core_hist[-1]:.4f}")

## 4. Amplitude Timeline

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    steps = np.arange(STEPS)

    for ax in axes:
        for x in [500, 1000, 1500]:
            ax.axvline(x, color="gray", linestyle="--", linewidth=0.5)

    axes[0].plot(steps, amp_core_hist, label="Core (layers 0-3)", linewidth=0.8)
    axes[0].plot(steps, amp_dom_hist, label="Domain knowledge", linewidth=0.8)
    axes[0].plot(steps, amp_mean_hist, label="Mean", linewidth=0.8, alpha=0.5)
    axes[0].set_ylabel("Amplitude")
    axes[0].legend(loc="lower right", fontsize=8)
    axes[0].set_title("Stuart-Landau Conviction Strength")
    for x, label in [
        (250, "supercritical"),
        (750, "subcritical (dk)"),
        (1250, "recovery"),
        (1750, "imprint"),
    ]:
        axes[0].text(
            x,
            axes[0].get_ylim()[1] * 0.98,
            label,
            ha="center",
            fontsize=8,
            style="italic",
        )

    axes[1].plot(steps, r_sl_hist, color="tab:purple", linewidth=0.8)
    axes[1].set_ylabel("R (amplitude-weighted)")
    axes[1].set_xlabel("Step")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## Summary

| Phase | Steps | R_good | R_domain | Amplitude |
|-------|-------|--------|----------|----------|
| Reconstruct | 0-499 | 0.23 -> 1.0 | 0.23 -> 1.0 | grows to ~1.18 |
| Disrupt | 500-999 | stays >0.88 | drops to ~0.16 | dk drops to ~0.48 |
| Repair | 1000-1499 | 1.0 | recovers to 1.0 | dk recovers |
| Imprint | 1500-1999 | 1.0 | 1.0 | stable at ~1.19 |

Core identity (working style, reasoning, relationship, aesthetics) is
**resilient** under domain-specific disruption. Domain knowledge is
**volatile** but **recoverable**. Imprint accumulates with sustained
coherence, strengthening coupling for future sessions.